# Result · Attacker-seed variance

**In the paper ([arXiv:2610.10899](https://arxiv.org/abs/2610.10899)):** the Sec. V paragraph "One attacker is not a measurement" and the 0.00–0.38 range of the abstract.

Five EnCodec attackers differing only in random seed against one fixed predictor (R0, R3, R3 multi; seed 0).
The spread between attackers exceeds every defence effect. Recomputes 3 jobs (~15 min each; stored attackers).

In [1]:
import os, sys
from pathlib import Path

# Find the repository root from wherever the notebook runs (notebooks/ or a subfolder).
ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "hackpatch").is_dir():
    if ROOT.parent == ROOT:
        raise RuntimeError("start the notebook from inside the repository")
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from IPython.display import Markdown, display
from hackpatch import repro

pd.set_option("display.width", 200, "display.max_columns", 40, "display.max_rows", 200)

In [2]:
# Parameters (also settable through environment variables for non-interactive runs)
# SOURCE = "recompute": re-run the evaluations behind this output from the released checkpoints (GPU), then aggregate.
# SOURCE = "released" : aggregate the released outputs of the original runs (CPU, seconds). This rewrites the
#                       committed analysis/out/ and paper/ files, which should come out identical.
SOURCE = os.environ.get("NB_SOURCE", "recompute")
# RUN = "slurm": one GPU job per run (sbatch), this kernel waits; "local": run in this kernel (needs a GPU).
RUN = os.environ.get("NB_RUN", "slurm")

In [3]:
reg = repro.registry()
KEYS = [k for k, v in reg.items() if v["kind"] in ['e1_attacker_seeds']]
print(len(KEYS), "runs:", ", ".join(KEYS))

3 runs: e1seed_R0_s0, e1seed_R3_s0, e1seed_R3ms_s0


In [4]:
if SOURCE == "recompute":
    repro.run_keys(KEYS, "recompute", RUN)  # skips runs whose recomputed output already exists
dirs = repro.run_analysis(SOURCE)  # whole analysis chain (seconds); falls back to released outputs where needed
OUT, PAPER = dirs["out"], dirs["paper"]
COMMITTED_OUT, COMMITTED_PAPER = ROOT / "analysis" / "out", ROOT / "paper"

[analysis] runs=/fs/scratch/PAS2301/alialavi/cache/audio_assessment_hackability_release/runs
[analysis] out=/users/PAS2301/alialavi/projects/audio_assessment_hackability/analysis/out
[analysis] paper=/users/PAS2301/alialavi/projects/audio_assessment_hackability/paper


In [5]:
# Where this notebook's inputs came from: recomputed from checkpoints, or released outputs of the original runs.
used = repro.load_registry(repro.results_registry(SOURCE))
pd.Series({k: used[k].get("source", "released") for k in KEYS}, name="source").value_counts()

source
released    3
Name: count, dtype: int64

In [6]:
av = pd.read_csv(OUT / "e1_attacker_variance.csv")
display(av.pivot(index="attacker", columns="target", values="asr_ref_recon").round(3))
av.groupby("target").asr_ref_recon.agg(["mean", "std", "min", "max"]).round(3)

target,R0,R3,R3ms
attacker,,,
a1,0.008,0.000,0.038
a2,0.225,0.382,0.080
a3,0.261,0.218,0.057
a4,0.000,0.001,0.086
a5,0.013,0.333,0.045


,mean,std,min,max
target,,,,
R0,0.102,0.130,0.000,0.261
R3,0.187,0.180,0.000,0.382
R3ms,0.061,0.021,0.038,0.086


In [7]:
repro.compare_csv(OUT / "e1_attacker_variance.csv", COMMITTED_OUT / "e1_attacker_variance.csv")

asr_ref_recon    0.0
asr_ref_input    0.0
Name: max_abs_diff, dtype: float64

In [8]:
# Numbers the paper quotes from this output: committed value vs this notebook's value.
cmp = repro.compare_numbers(OUT / "paper_numbers.json", prefixes=('av_',))
print(f"{len(cmp)} numbers; max |diff| = {cmp.abs_diff.max():.4f}; identical: {(cmp.abs_diff == 0).sum()}")
cmp.head(20)

24 numbers; max |diff| = 0.0000; identical: 24


,paper,reproduced,abs_diff
av_R0_max,0.261,0.261,0.0
av_R0_max_2dp,0.260,0.260,0.0
av_R0_mean,0.102,0.102,0.0
av_R0_mean_2dp,0.100,0.100,0.0
av_R0_min,0.000,0.000,0.0
av_R0_min_2dp,0.000,0.000,0.0
av_R0_sd,0.130,0.130,0.0
av_R0_sd_2dp,0.130,0.130,0.0
av_R3_max,0.382,0.382,0.0
av_R3_max_2dp,0.380,0.380,0.0
